# Análise de mecanismos de controle em RAG

**DCA0305 / UFRN · rag-control-lab**

Este caderno apresenta a execução de 20 perguntas nas variantes LLM puro,
baseline, Adaptive e Self-RAG. O foco é seguir **lacuna → decisão → evidência → efeito → custo**.
Os cinco TCCs são o corpus de teste; o objeto do estudo são os mecanismos de controle.

**Fluxo padrão:** analisar os resultados já salvos, sem chamar modelos, consultar o Ollama
ou ler PDFs. A reprodução de inferência fica ao final, desligada por padrão.


## 1. Abrir o projeto e selecionar a rodada


In [1]:
from pathlib import Path
import json
import sys

candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/rag-control-lab')]
ROOT = next((p for p in candidates if (p / 'configs/analysis.json').is_file()), None)
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))

RUN_TO_ANALYZE = None          # None usa configs/analysis.json.
EXAMPLE_QUESTION = 'Q02'       # Troque para Q04, Q07, Q14, Q18 etc.
EXPORT_FIGURES = False        # Exportação opcional de PNG/SVG, sem alterar os dados brutos.
ENABLE_INFERENCE = False
PREPARE_CORPUS = False
BUILD_INDEX = False
RUN_EXPERIMENT = False
EXPERIMENT_MODE = 'controlled'

import pandas as pd
from IPython.display import display, Markdown
from reporting import load_analysis, example, plot_results, LABELS

report = load_analysis(ROOT, RUN_TO_ANALYZE)
run_dir = report['run_dir']
print('Projeto:', ROOT.name)
print('Rodada:', run_dir.name)
print('Estado:', report['manifest']['status'], '| condição:', report['manifest']['evaluation_mode'])
print('Respostas únicas:', len(report['answers']))
print('Inferência habilitada:', ENABLE_INFERENCE)


Projeto: rag-control-lab
Rodada: 20261018T182531677492Z_81b2dd68bc
Estado: complete | condição: controlled
Respostas únicas: 80
Inferência habilitada: False


## 2. Pergunta de pesquisa e variantes

**Em quais situações a busca adaptativa e a verificação tratam limitações de uma
recuperação única, e qual esforço adicional exigem?**

| Variante | Comportamento implementado |
| --- | --- |
| LLM puro | Geração sem documentos. |
| Baseline | Geração com o pacote inicial controlado. |
| Adaptive | Planejamento, seleção e buscas complementares por lacunas antes da geração. |
| Self-RAG | Rascunho, verificação de apoio/cobertura, busca e revisão dos itens selecionados. |


## 3. Corpus e configuração registrados

As informações vêm do manifesto da rodada. Ler os resultados não executa ingestão,
não reconstrói embeddings e não modifica `corpus/manifest.csv`. 

In [2]:
documents = pd.DataFrame(report['manifest']['signature']['corpus'])
display(documents[['doc_id', 'title', 'pages', 'text_pages', 'source_url']])
cfg = report['manifest']['signature']['config']
parameters = ['generator_model', 'embedding_model', 'retriever', 'chunk_size_chars',
              'chunk_overlap_chars', 'top_k', 'context_char_cap', 'num_ctx', 'temperature', 'seeds']
display(pd.DataFrame([{'parâmetro': k, 'valor': str(cfg[k])} for k in parameters]))
print('Páginas físicas:', int(documents.pages.sum()))
print('Corpus:', report['manifest']['signature']['questions']['corpus_fingerprint'])


,doc_id,title,pages,text_pages,source_url
0,DOC01,Desenvolvimento de um Sistema Interativo para ...,55,46,https://repositorio.ufrn.br/handle/123456789/6...
1,DOC02,Sistema de rastreamento solar de dois eixos ba...,55,55,https://repositorio.ufrn.br/handle/123456789/6...
2,DOC03,Sistema Assistivo Multiagente para Gestão de R...,46,46,https://repositorio.ufrn.br/handle/123456789/6...
3,DOC04,Navegação autônoma local: estudo empírico sobr...,80,80,https://repositorio.ufrn.br/handle/123456789/6...
4,DOC05,Análise de escalabilidade de um algoritmo de m...,50,42,https://repositorio.ufrn.br/handle/123456789/7...


,parâmetro,valor
0,generator_model,qwen2.5:3b
1,embedding_model,embeddinggemma:300m
2,retriever,hybrid
3,chunk_size_chars,900
4,chunk_overlap_chars,150
5,top_k,4
6,context_char_cap,6500
7,num_ctx,4096
8,temperature,0.0
9,seeds,[42]


Páginas físicas: 286
Corpus: f4c93f2fcf9c2e9efbba4a684194ad9e2c6480605187c355bf8f116ef2a306dd


## 4. Perguntas e condições de avaliação

Há uma pergunta de cada cenário por documento. Na Condição A, as variantes RAG
recebem o mesmo pacote inicial; buscas extras usam o corpus comum. O LLM puro
continua sem documentos. A Condição B usa recuperação natural, mas não foi avaliada aqui.

`direct`, `two_step`, `distractor` e `unanswerable` são rótulos planejados.


In [3]:
questions = report['questions']
display(pd.crosstab(questions['doc_id'], questions['scenario']))
display(questions[['id', 'doc_id', 'scenario', 'question', 'controlled_initial_chunks']])


scenario,direct,distractor,two_step,unanswerable
doc_id,,,,
DOC01,1,1,1,1
DOC02,1,1,1,1
DOC03,1,1,1,1
DOC04,1,1,1,1
DOC05,1,1,1,1


,id,doc_id,scenario,question,controlled_initial_chunks
0,Q01,DOC01,direct,No dashboard da Biblioteca Padre Jaime Diniz (...,[DOC01:P0027:C0002]
1,Q02,DOC01,two_step,"No projeto da Biblioteca (DOC01), como o Gspre...",[DOC01:P0027:C0001]
2,Q03,DOC01,distractor,"No dashboard da biblioteca (DOC01), como o car...","[DOC01:P0028:C0002, DOC01:P0036:C0002]"
3,Q04,DOC01,unanswerable,No sistema da Biblioteca Padre Jaime Diniz (DO...,"[DOC01:P0027:C0002, DOC01:P0053:C0001]"
4,Q05,DOC02,direct,"No projeto do forno solar (DOC02), qual softwa...",[DOC02:P0029:C0001]
5,Q06,DOC02,two_step,"No forno solar (DOC02), qual particularidade m...",[DOC02:P0029:C0001]
6,Q07,DOC02,distractor,"No forno solar (DOC02), por que a variante de ...","[DOC02:P0025:C0002, DOC02:P0030:C0001]"
7,Q08,DOC02,unanswerable,"No protótipo de forno solar (DOC02), qual temp...","[DOC02:P0034:C0001, DOC02:P0037:C0001]"
8,Q09,DOC03,direct,"No Organiz.ai (DOC03), qual framework implemen...",[DOC03:P0025:C0001]
9,Q10,DOC03,two_step,"No Organiz.ai (DOC03), qual é o papel do orque...",[DOC03:P0025:C0002]


## 5. Encontrar, entregar e usar evidência

Uma busca útil pode encontrar informação que o seletor descarta. Um contexto útil
pode chegar ao gerador sem ser interpretado corretamente. As três etapas não são
equivalentes. As anotações de mecanismos estão em `review.json` e foram
conferidas contra os rastros; sua consolidação não chama o LLM.


In [4]:
columns = ['variant', 'questions', 'questions_with_extra_search',
           'questions_with_useful_retrieval', 'questions_with_useful_delivery']
display(report['control_summary'][columns])
display(Markdown('**Perguntas de duas etapas:** algum complemento encontrado/entregue e contexto suficiente para todos os critérios.'))
display(report['two_step'])
figures = plot_results(report, ROOT / 'results/figures' if EXPORT_FIGURES else None)
import matplotlib.pyplot as plt
for figure in figures.values():
    plt.close(figure)
display(figures['control_evidence'])


,variant,questions,questions_with_extra_search,questions_with_useful_retrieval,questions_with_useful_delivery
0,adaptive,20,19,10,5
1,self_rag,20,17,10,6


**Perguntas de duas etapas:** algum complemento encontrado/entregue e contexto suficiente para todos os critérios.

,variant,questions,useful_retrieval,useful_delivery,complete_context
0,adaptive,5,5,4,1
1,self_rag,5,5,5,3


<Figure size 900x470 with 1 Axes>

## 6. Rastreabilidade da retomada

O fingerprint observado antes da retomada mudou, e os eventos comprovam duas formas
de revisão: vários itens em uma chamada e uma chamada por item. O código atual usa
a segunda.

Com um único item alvo, os eventos não permitem distinguir as duas políticas. Os
custos são descrições do que foi executado, não uma comparação uniforme de políticas.


In [5]:
modes = report['revision_modes']
display(modes[['question_id', 'revision_mode_observed', 'revision_targets',
               'generator_calls', 'items_per_generator_call']])
display(modes.groupby('revision_mode_observed').agg(
    perguntas=('question_id', 'size'), chamadas_médias=('llm_calls', 'mean'),
    latência_média_s=('latency_seconds', 'mean')).reset_index())


,question_id,revision_mode_observed,revision_targets,generator_calls,items_per_generator_call
0,Q01,no_revision,0,1,[1]
1,Q02,joint_revision,2,2,"[2, 2]"
2,Q03,joint_revision,2,2,"[4, 2]"
3,Q04,single_target_ambiguous,1,2,"[1, 1]"
4,Q05,no_revision,0,1,[1]
5,Q06,joint_revision,4,2,"[5, 4]"
6,Q07,joint_revision,2,2,"[5, 2]"
7,Q08,joint_revision,2,2,"[2, 2]"
8,Q09,no_revision,0,1,[1]
9,Q10,per_item_revision,5,6,"[5, 1, 1, 1, 1, 1]"


,revision_mode_observed,perguntas,chamadas_médias,latência_média_s
0,joint_revision,5,20.80,159.920809
1,no_revision,3,5.00,19.256694
2,per_item_revision,8,31.25,253.139926
3,single_target_ambiguous,4,9.75,78.583922


## 7. Inspecionar um caso: pergunta, rascunho e evidências

Altere `EXAMPLE_QUESTION` na primeira célula para comparar outros casos. Q02 mostra
correção do motivo da Drive API; Q04 mostra uma recusa adequada convertida em
latência inventada; Q07 mostra cobertura com evidência já disponível; Q18 mostra
busca útil e informação correta que não corrige uma estratégia previamente aprovada.


In [6]:
question, answers, stages, evidence = example(report, EXAMPLE_QUESTION)
display(Markdown('### ' + EXAMPLE_QUESTION))
display(Markdown(question))
with pd.option_context('display.max_colwidth', None):
    display(answers)
    display(stages)
display(evidence)
display(report['control_cases'].loc[
    report['control_cases'].question_id.eq(EXAMPLE_QUESTION),
    ['variant', 'initial_criteria', 'final_context_criteria', 'new_retrieved_criteria',
     'new_delivered_criteria', 'effect', 'note']])


### Q02

No projeto da Biblioteca (DOC01), como o Gspread autentica o acesso à planilha e por que a Google Drive API precisou ser ativada junto à Google Sheets API?

,variant,answer,abstained
1,pure_llm,"No projeto da Biblioteca (DOC01), como o Gspread autentica o acesso à planilha e por que a Google Drive API precisou ser: O Gspread autentica o acesso à planilha usando a autenticação OAuth 2.0 da Google. A Google Drive API foi ativada junto à Google Sheets API para fornecer acesso à planilhas no Google Drive.",False
21,baseline,"Como o Gspread autentica o acesso à planilha?: Gspread autentica o acesso à planilha usando uma conta de serviço criada com papel de leitor/editor, que representa a aplicação perante a API do Google, sem exigir login interativo de um usuário real.\nPor que a Google Drive API precisou ser ativada junto à Google Sheets API?: A Google Drive API precisou ser ativada junto à Google Sheets API para permitir que Gspread compartilhe a planilha de controle da Biblioteca Padre Jaime Diniz com a conta de serviço criada, sem o compartilhamento explícito, a API retorna erro de acesso negado mesmo com credenciais válidas.",False
41,adaptive,"Descrição da autenticação de acesso à planilha usando Gspread no projeto da Biblioteca (DOC01): A autenticação de acesso à planilha usando Gspread no projeto da Biblioteca (DOC01) é realizada através da criação de uma conta de serviço com papel de leitor/editor, que representa a aplicação perante a API do Google, sem exigir login interativo de um usuário real. Além disso, é necessário compartilhar a planilha com o endereço de e-mail da conta de serviço e gerar uma chave no formato JSON para o arquivo credenciais.json.\nMotivo da necessidade de atividade da Google Drive API junto à Google Sheets API no projeto da Biblioteca (DOC01): A atividade da Google Drive API junto à Google Sheets API no projeto da Biblioteca (DOC01) é necessária para compartilhar a planilha com o endereço de e-mail da conta de serviço, o que permite que a aplicação se autentique perante os servidores do Google e acesse os dados necessários de maneira 100% automatizada.",False
61,self_rag,"Descrição da autenticação de acesso à planilha usando Gspread no projeto da Biblioteca (DOC01): O Gspread autentica o acesso à planilha criando uma conta de serviço com papel de leitor/editor, que representa a aplicação perante a API do Google, sem exigir login interativo de um usuário real.\nMotivo da necessidade de atividade da Google Drive API junto à Google Sheets API no projeto da Biblioteca (DOC01): A atividade da Google Drive API junto à Google Sheets API é necessária porque a biblioteca Gspread utiliza internamente a Drive API para localizar a planilha pelo nome antes de acessá-la pela Sheets API.",False


,variant,stage,answer
0,adaptive,1,"Descrição da autenticação de acesso à planilha usando Gspread no projeto da Biblioteca (DOC01): A autenticação de acesso à planilha usando Gspread no projeto da Biblioteca (DOC01) é realizada através da criação de uma conta de serviço com papel de leitor/editor, que representa a aplicação perante a API do Google, sem exigir login interativo de um usuário real. Além disso, é necessário compartilhar a planilha com o endereço de e-mail da conta de serviço e gerar uma chave no formato JSON para o arquivo credenciais.json.\nMotivo da necessidade de atividade da Google Drive API junto à Google Sheets API no projeto da Biblioteca (DOC01): A atividade da Google Drive API junto à Google Sheets API no projeto da Biblioteca (DOC01) é necessária para compartilhar a planilha com o endereço de e-mail da conta de serviço, o que permite que a aplicação se autentique perante os servidores do Google e acesse os dados necessários de maneira 100% automatizada."
1,self_rag,1,"Descrição da autenticação de acesso à planilha usando Gspread no projeto da Biblioteca (DOC01): O Gspread autentica o acesso à planilha criando uma conta de serviço com papel de leitor/editor, que representa a aplicação perante a API do Google, sem exigir login interativo de um usuário real.\nMotivo da necessidade de atividade da Google Drive API junto à Google Sheets API no projeto da Biblioteca (DOC01): A atividade da Google Drive API junto à Google Sheets API é necessária para compartilhar a planilha com a conta de serviço, o que é indispensável para evitar erro de acesso negado mesmo com credenciais válidas."
2,self_rag,2,"Descrição da autenticação de acesso à planilha usando Gspread no projeto da Biblioteca (DOC01): O Gspread autentica o acesso à planilha criando uma conta de serviço com papel de leitor/editor, que representa a aplicação perante a API do Google, sem exigir login interativo de um usuário real.\nMotivo da necessidade de atividade da Google Drive API junto à Google Sheets API no projeto da Biblioteca (DOC01): A atividade da Google Drive API junto à Google Sheets API é necessária porque a biblioteca Gspread utiliza internamente a Drive API para localizar a planilha pelo nome antes de acessá-la pela Sheets API."


,variant,source_id,page,text
0,adaptive,DOC01:P0027:L913-1074,27,3.2.2 Função: buscar_dados_brutos_da_planilha(...
1,adaptive,DOC01:P0027:L365-367,27,5.
2,adaptive,DOC01:P0023:L1065-1255,23,Isso permite que a aplicação se autentique per...
3,adaptive,DOC01:P0027:L368-563,27,Compartilhamento da planilha com a conta de se...
4,adaptive,DOC01:P0027:L25-242,27,"Criação da conta de serviço: em ""APIs e Serviç..."
5,adaptive,DOC01:P0027:L564-699,27,"Esse passo é indispensável, pois sem o compar-..."
6,adaptive,DOC01:P0027:L246-364,27,Download do arquivo de credenciais: após criar...
7,adaptive,DOC01:P0027:L700-900,27,"Com essa configuração estabelecida, a bibliote..."
8,self_rag,DOC01:P0027:C0001,27,3.2. CAMADA DE DADOS7 3. Criação da conta de s...
9,self_rag,DOC01:P0027:L913-1074,27,3.2.2 Função: buscar_dados_brutos_da_planilha(...


,variant,initial_criteria,final_context_criteria,new_retrieved_criteria,new_delivered_criteria,effect,note
2,adaptive,"[FULL, ABSENT]","[FULL, ABSENT]",[2],[],mixed,Busca de autenticação recupera a p.26 com o mo...
3,self_rag,"[FULL, ABSENT]","[FULL, FULL]",[2],[2],benefit,"Detecta motivo sem apoio, recupera p.26 e corr..."


## 8. Verificação e escopo do planejamento

Apoio e cobertura são avaliados separadamente. Uma afirmação correta pode
estar incompleta. Os pareceres indeterminados não contam como acertos ou erros;
`epascal_stop` é tratado como artefato de extração, separado do apoio literal ao texto.
Itens repetidos e etapas da mesma pergunta não são observações independentes.

Necessidades inventadas pelo planejador não viram novas obrigações da pergunta.
Uma recusa a item extra pode inflar `abstained=True`, mesmo com a resposta pedida correta.


In [7]:
checks = report['checks']
display(checks.groupby('support_classification').size().rename('ocorrências').reset_index())
scope = pd.DataFrame(report['review']['control_cases'])
display(scope[['question_id', 'unrequested_needs', 'redundant_needs', 'omitted_requested_aspects']])
print('Pareceres factuais examinados:', len(checks))


,support_classification,ocorrências
0,appropriate_flag,30
1,correct_approval,25
2,false_approval,10
3,false_rejection,16
4,text_supported_extraction_error,2
5,undetermined,28


,question_id,unrequested_needs,redundant_needs,omitted_requested_aspects
0,Q01,[],[],
1,Q02,[],[],
2,Q03,[],"[3, 4]",
3,Q04,[],[],
4,Q05,[],[],
5,Q06,"[3, 4]",[5],
6,Q07,[5],[],
7,Q08,[],[],
8,Q09,[],[],
9,Q10,[],"[3, 4, 5]",


Pareceres factuais examinados: 111


## 9. Custo observado

As chamadas de planejamento, seleção, verificação, preservação e geração entram
na contagem. No baseline controlado, zero buscas extras é esperado porque o contexto
inicial é fornecido. As médias não incluem todo o tempo de desenvolvimento ou as
tentativas interrompidas. O Self-RAG combina revisões em lote e por item.

Taxa de abstenção declarada significa algum item recusado, inclusive itens extras.
Não representa segurança nem acerto na decisão de recusar.


In [8]:
display(report['process'])
display(figures['process_cost'])
display(report['answers'].groupby(['variant', 'scenario']).agg(
    respostas=('question_id', 'size'), chamadas_LLM=('llm_calls', 'mean'),
    latência_s=('latency_seconds', 'mean'), algum_item_recusado=('abstained', 'sum')).reset_index())


,variant,answers,llm_calls_mean,extra_searches_mean,latency_mean_seconds,declared_any_item_abstentions
0,pure_llm,20,1.00,0.00,5.116284,9
1,baseline,20,1.00,0.00,13.038466,8
2,adaptive,20,7.05,1.90,70.194543,10
3,self_rag,20,20.40,1.85,159.841461,13


<Figure size 1000x480 with 2 Axes>

,variant,scenario,respostas,chamadas_LLM,latência_s,algum_item_recusado
0,adaptive,direct,5,4.6,35.363908,1
1,adaptive,distractor,5,9.0,93.251877,4
2,adaptive,two_step,5,9.0,106.136543,2
3,adaptive,unanswerable,5,5.6,46.025844,3
4,baseline,direct,5,1.0,8.678489,0
5,baseline,distractor,5,1.0,17.013569,0
6,baseline,two_step,5,1.0,16.631598,3
7,baseline,unanswerable,5,1.0,9.830206,5
8,pure_llm,direct,5,1.0,3.932236,4
9,pure_llm,distractor,5,1.0,5.613599,0


## 10. Qualidade final 

Acerto integral exige todos os critérios
e ausência de erro material; cobertura parcial não é acerto integral. O ganho de uma
etapa não demonstra melhoria da resposta completa.


In [ ]:
quality = report['quality']
display(quality.groupby('variant').agg(
    respostas=('question_id', 'size'), acertos_integrais=('correct_integral', 'sum'),
    fração_média_de_critérios=('criteria_met_fraction', 'mean'),
    respostas_com_erro_material=('material_error', 'sum')).reindex(LABELS))
print(f'Respostas com avaliação registrada: {len(quality)}/{len(report["answers"])}.')


,respostas,acertos_integrais,fração_média_de_critérios,respostas_com_erro_material
variant,,,,
pure_llm,20,2,0.125000,8
baseline,20,8,0.550000,5
adaptive,20,4,0.358333,12
self_rag,20,7,0.566667,8
